# Notebook 14 — Which genes we test for selection, and why

**Purpose.** This notebook is the *gene-panel justification* for the primate sexual-dichromatism selection
scan. For every gene whose selection we test across primates, it records **where we know it as a pigmentation
or sex-hormone gene from** — drawn only from **our own network layers** and the pigmentation references already
in this repository. It documents both the genes **already analyzed** (80: 53 hormone + 27 pigmentation) and a
principled **expansion of the pigmentation module** (43 genes) that brings the two modules to comparable
pathway breadth.

**Why this notebook exists.** The original panel was built asymmetrically. The **hormone module** was assembled
as a *whole endocrine pathway* (53 genes: steroid biosynthesis, HPG axis, receptors, coactivators). The
**pigmentation module** was only the *canonical melanogenesis core* (27 genes: the synthesis enzymes, a few
transporters and transcription factors). Because there are ~2x as many hormone genes, raw per-module
counts are hormone-heavy by construction, and any "module-balance" metric is biased. The fix is not to trim
hormones but to **expand pigmentation to the same breadth** — using genes that are pigmentation genes *in our
own network layers*, not by importing a new external list.

**Evidence sources (all already in this repo).**
- **Raghunath melanogenesis mechanism** — the curated melanin-synthesis pathway layer (the user's standard:
  melanogenesis-pathway membership counts as pigmentation evidence).
- **OMIM pigmentation phenotype** — a gene whose loss/variation causes a documented human pigmentation disorder
  (albinism, Hermansky-Pudlak, Griscelli, vitiligo, hyper/hypopigmentation), annotated on our 803-node network.
- **Baxter et al. 2019** — the curated reference set of pigmentation genes (Baxter et al. 2019, *Pigment Cell
  Melanoma Res* 32:348, doi:10.1111/pcmr.12743), 657 human symbols.
- **Melanosome proteome (mass-spec)** and **functional CRISPR screen (Bajpai 2023)** — corroborating layers.

**What we do NOT add.** Genes that entered the network only through generic STRING/KEGG proximity with no
pigmentation phenotype (e.g. pan-cellular kinases, apoptosis machinery) are excluded — adding them would repeat
the breadth error in a subtler form.

In [ ]:
import os, pandas as pd
REPO = os.environ.get("PIGNET_REPO", os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")))
CG   = os.path.join(REPO, "comparative-genomics")
DATA = os.path.join(CG, "analysis", "module_selection", "data")
J = pd.read_csv(os.path.join(DATA, "nb14_panel_justification.csv"))
print(f"genes documented: {len(J)}  |  already run: {(J.status=='already_run').sum()}  |  to run: {(J.status=='to_run').sum()}")
print("by module:", J.module.value_counts().to_dict())

## 1 — The sex-hormone module (53 genes, already analyzed)

Justification = position in the endocrine pathway. The module spans the full hypothalamic-pituitary-gonadal
(HPG) axis and steroid metabolism, because sexual dichromatism is a sex-limited trait and the a-priori
hypothesis is that sex-steroid signaling gates its expression.

In [ ]:
hor = J[J.module == "hormone"]
print(hor.category.value_counts().to_string())
print("\ngenes:", ", ".join(sorted(hor.gene)))

## 2 — The pigmentation module, as originally run (27 genes)

The canonical melanogenesis core: synthesis enzymes (TYR, TYRP1, DCT), melanosome structural/transport
proteins, the MC1R/ASIP/POMC receptor-signaling module, and the MITF/SOX10/PAX3/TFAP2A transcription factors.
Every one is Baxter-listed and/or has an OMIM pigmentation phenotype.

In [ ]:
pig_run = J[(J.module == "pigmentation") & (J.status == "already_run")]
print(pig_run[["gene", "category", "justification"]].to_string(index=False))

## 3 — The pigmentation expansion (43 genes, to be analyzed)

Two groups, both drawn from our network layers:

**(a) Melanosome biogenesis & transport (21 genes)** — the single largest omission. These are the
albinism / Hermansky-Pudlak / Griscelli / vitiligo genes that build melanosomes and move them into
keratinocytes: HPS1/3/4/5/6, the AP-3 and BLOC-1 complexes, the Griscelli transport tripod (RAB27A, MYO5A,
MLPH), LYST, GPR143. The original panel stopped at melanin *synthesis* and omitted melanosome *biology*
entirely. Every gene here has an OMIM pigmentation phenotype.

**(b) Melanogenesis regulation (22 genes)** — the MAPK / Wnt / endothelin signaling arm that regulates MITF,
from the Raghunath melanogenesis-mechanism layer, each with corroborating pigmentation-specific evidence
(OMIM pigment phenotype, Baxter, or functional screen): MAP2K1/2, RAF1, HRAS, MAPK3, CTNNB1, EDN1, HGF, PAK1,
CDC42, and related regulators.

In [ ]:
add = J[J.status == "to_run"]
for grp in ["melanosome_biogenesis", "melanogenesis_regulation"]:
    sub = add[add.category == grp]
    print(f"\n{grp} ({len(sub)}):")
    print(sub[["gene", "justification"]].to_string(index=False))

## 4 — Effect on module balance

The expansion brings the pigmentation module from 27 to 70 genes, against 53 hormone genes — so the panel
becomes pigmentation-weighted, matching the biology (pigmentation is the larger, better-characterized network).
After the expansion runs, the module-balance metric should be recomputed as a **per-gene selection rate**
(selected / genes-tested, per module) so it is not sensitive to how many genes each module happens to contain.

In [ ]:
n_pig_before, n_hor = 27, 53
n_pig_after = (J[J.module == "pigmentation"]).gene.nunique()
print(f"pigmentation module: {n_pig_before} -> {n_pig_after} genes")
print(f"hormone module:      {n_hor} genes (unchanged)")
print(f"panel-neutral balance point moves from {2*n_pig_before/(n_pig_before+n_hor)-1:+.2f} "
      f"toward {2*n_pig_after/(n_pig_after+n_hor)-1:+.2f}")
print("=> after expansion, report module balance as a per-gene RATE, not a raw count difference.")

## Result

A documented, reproducible panel: **123 genes** total, each traceable to a pigmentation or hormone evidence
source in this repository. 80 already analyzed; 43 pigmentation genes queued for the selection scan. The full
table is `data/nb14_panel_justification.csv`; the expansion list for the cluster is
`config/gene_panel_expansion_pigmentation.csv`.